In [15]:
!pip install simpletransformers

In [18]:
import json
with open(r"train.json", "r") as read_file:
  train = json.load(read_file)

In [19]:
train

[{'context': 'Mistborn is a series of epic fantasy novels written by American author Brandon Sanderson.',
  'qas': [{'id': '00001',
    'is_impossible': False,
    'question': 'Who is the author of the Mistborn series?',
    'answers': [{'text': 'Brandon Sanderson', 'answer_start': 71}]}]},
 {'context': 'The first series, published between 2006 and 2008, consists of The Final Empire,The Well of Ascension, and The Hero of Ages.',
  'qas': [{'id': '00002',
    'is_impossible': False,
    'question': 'When was the series published?',
    'answers': [{'text': 'between 2006 and 2008', 'answer_start': 28}]},
   {'id': '00003',
    'is_impossible': False,
    'question': 'What are the three books in the series?',
    'answers': [{'text': 'The Final Empire, The Well of Ascension, and The Hero of Ages',
      'answer_start': 63}]},
   {'id': '00004',
    'is_impossible': True,
    'question': 'Who is the main character in the series?',
    'answers': []}]}]

In [20]:
with open(r"test.json", "r") as read_file:
  test = json.load(read_file)

In [21]:
test

[{'context': 'The series primarily takes place in a region called the Final Empire on a world called Scadrial, where the sun and sky are red, vegetation is brown, and the ground is constantly being covered under black volcanic ashfalls.',
  'qas': [{'id': '00001',
    'is_impossible': False,
    'question': 'Where does the series take place?',
    'answers': [{'text': 'region called the Final Empire', 'answer_start': 38},
     {'text': 'world called Scadrial', 'answer_start': 74}]}]},
 {'context': '"Mistings" have only one of the many Allomantic powers, while "Mistborns" have all the powers.',
  'qas': [{'id': '00002',
    'is_impossible': False,
    'question': 'How many powers does a Misting possess?',
    'answers': [{'text': 'one', 'answer_start': 21}]},
   {'id': '00003',
    'is_impossible': True,
    'question': 'What are Allomantic powers?',
    'answers': []}]}]

In [22]:
import logging
from simpletransformers.question_answering import QuestionAnsweringModel, QuestionAnsweringArgs

In [23]:
model_type = "bert"
model_name = "bert-base-cased"
if model_type == "bert":
  model_name = "bert-base-cased"
elif model_type == "roberta":
  model_name = "roberta-base"
elif model_type == "distilbert":
  model_name = "distilbert-base-cased"
elif model_type == "distilroberta":
  model_type = "roberta"
  model_name = "distilroberta-base"
elif model_type == "electra-base":
  model_type = "electra"
  model_name = "google/electra-base-discriminator"
elif model_type == "electra-small":
  model_type = "electra"
  model_name = "google/electra-small-discriminator"
elif model_type == "xlnet":
  model_name = "xlnet-base-cased"

In [24]:
model_args = QuestionAnsweringArgs()
model_args.train_batch_size = 16
model_args.evaluate_durin_training = True
model_args.n_best_size = 3
model_args.num_training_epochs = 5

In [25]:
!pip install wandb

In [48]:
train_args = {
    "reprocess_input_data" : True,
    "overwrite_output_dir" : True,
    "use_cached_eval_features" : True,
    "output_dir" : f"outputs/{model_type}",
    "best_model_dir" : f"outputs/{model_type}/best_model",
    "evaluate_during_training" : True,
    "max_sequence_length" : 128,
    "num_training_epochs" : 10,
    "evaluate_during_training_steps" : 1000,
    "wandb_project" : "QA-Appln-1",
    "wandb_kwargs" : {"name" : model_name},
    "save_model_every_epoch" : False,
    "save_eval_checkpoints" : False,
    "n_best_size" : 3,
    #"use_early_stopping" : True,
    #"early_stopping_metric" : "mcc",
    #"n_gpu" : 2,
    #"manual_seed" : 4,
    #"use_multiprocessing" : False,
    "train_batch_size" : 128,
    "test_batch_size" : 64,
    #"config" : {
        #"output_hidden_states" : True
    #}
}

In [49]:
model =  QuestionAnsweringModel(
    model_type, model_name, args = train_args
)

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [50]:
# To remove o/p folder
!rm -rf outputs
!rm -rf cache_dir
!rm -rf runs
!rm -rf wandb

In [51]:
model.train_model(train, eval_data = test)

add example index and unique id: 100%|██████████| 4/4 [00:00<00:00, 19878.22it/s]


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

correct,▁
eval_loss,▁
global_step,▁
incorrect,▁
similar,▁
train_loss,▁
correct,1
eval_loss,0.06506
global_step,1
incorrect,0
similar,2


Running Epoch 1 of 1:   0%|          | 0/1 [00:00<?, ?it/s]


convert squad examples to features: 100%|██████████| 3/3 [00:00<00:00, 217.57it/s]

add example index and unique id: 100%|██████████| 3/3 [00:00<00:00, 15907.60it/s]


Running Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

(1,
 {'global_step': [1],
  'correct': [0],
  'similar': [1],
  'incorrect': [2],
  'train_loss': [4.856770992279053],
  'eval_loss': [0.08740234375]})

In [52]:
result, texts = model.eval_model(test)

Running Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

In [53]:
result

{'correct': 0, 'similar': 1, 'incorrect': 2, 'eval_loss': 0.08740234375}

In [54]:
texts

{'correct_text': {},
 'similar_text': {'00003': {'truth': '',
   'predicted': 'Allomantic powers, while "Mist',
   'question': 'What are Allomantic powers?'}},
 'incorrect_text': {'00001': {'truth': 'region called the Final Empire',
   'predicted': 'sky are red, vegetation is brown, and the ground is constantly being',
   'question': 'Where does the series take place?'},
  '00002': {'truth': 'one',
   'predicted': 'Allomantic powers, while "Mist',
   'question': 'How many powers does a Misting possess?'}}}

In [55]:
to_predict = [
    {
        "context" : "Iron Maiden is a English heavy metal band from Leyton, East London.",
        "qas" : [
            {
                "question" : "Where is Iron Maiden from?",
                "id" : "0",
            }
        ],
    }
]

In [56]:
answers, probabilities = model.predict(to_predict)
print(answers)

add example index and unique id: 100%|██████████| 1/1 [00:00<00:00, 4993.22it/s]


Running Prediction:   0%|          | 0/1 [00:00<?, ?it/s]

[{'id': '0', 'answer': ['heavy metal band from Leyton, East London', 'heavy metal band from Leyton']}]


In [62]:
to_predict1 = [
    {
        "context" : """ It is natural to want to employ your friends when you find yourself in times
 of need. The world is a harsh place, and your friends soften the harshness.
 Besides, you know them. ‘Why depend on a stranger when you have a
 friend at hand?
 The problem is that you often do not know your friends as well as you
 imagine you do. Friends often agree on things in order to avoid an argument.
 They cover up their unpleasant qualities so as to not offend each other.
 They laugh extra hard at each other’s jokes. Since honesty rarely strengthens friendship,
 you may never know how a friend truly feels. Friends will
 say that they love your poetry, adore your music, envy your taste in
 clothes - maybe they mean it, often they do not.
 When you decide to hire a friend, you gradually discover the qualities
 he or she has kept hidden. Strangely enough, it is your act of kindness that
 unbalances everything. People want to feel they deserve their good fortune.
 The receipt of a favor can become oppressive: It means you have
 been chosen because you are a friend, not necessarily because you are deserving.
 There is almost a touch of condescension in the act of hiring
 friends that secretly afflicts them. The injury will come out slowly: A little
 honesty, flashes of resentment and envy here and there, and before
 you know it your friendship fades. The more favors and gifts you supply to
 revive the friendship, the less gratitude you receive. The problem with using or hiring friends is that it will inevitably limit
 your power. The friend is rarely the one who is most able to help you; and
 in the end, skill and competence are far more important than friendly feelings.
 (Michael Ill had a man right under his nose who would have steered
 him right and kept him alive: That man was Bardas.)
 All working situations require a kind of distance between people. You
 are trying to work, not make friends; friendliness (real or false) only obscures that fact.
 The key to power, then, is the ability to judge who is best
 able to further your interests in all situations. Keep friends for friendship,
 but work with the skilled and competent.
 Your enemies, on the other hand, are an untapped gold mine that you
 must learn to exploit. When Talleyrand, Napoleon’s foreign minister, decided
 in 1807 that his boss was leading France to ruin, and the time had
 come to turn against him, he understood the dangers of conspiring against
 the emperor; he needed a partner, a confederate - what friend could he
 trust in such a project? He chose Joseph Fouché, head ofthe secret police,
 his most hated enemy, a man who had even tried to have him assassinated.
 He knew that their former hatred would create an opportunity for an emotional reconciliation.
 He knew that Fouché would expect nothing from
 him, and in fact would work to prove that he was worthy of Talleyrand’s
 choice; a person who has something to prove will move mountains for you.
 Finally, he knew that his relationship with Fouché would be based on mutual
 self interest, and would not be contaminated by personal feelings. The
 selection proved perfect; although the conspirators did not succeed in toppling
 Napoleon, the union of such powerful but unlikely partners generated
 much interest in the cause; opposition to the emperor slowly began to
 spread. And from then on, Talleyrand and Fouché had a fruitful working
 relationship. Whenever you can, bury the hatchet with an enemy, and
 make a point of putting him in your service.
 As Lincoln said, you destroy an enemy when you make a friend of
 him. In 1971, during the Vietnam War, Henry Kissinger was the target of
 an unsuccessful kidnapping attempt, a conspiracy involving, among others,
 the renowned anti-war activist priests the Berrigan brothers, four more
 Catholic priests, and four nuns. In private, without informing the Secret
 Service or the justice Department, Kissinger arranged a Saturday morning
 meeting with three of the alleged kidnappers. Explaining to his guests that
 he would have most of the American soldiers out of Vietnam by mid 1972, he
 completely charmed them. They gave him some “Kidnap Kissinger” buttons
 and one of them remained a friend of his for years, visiting him on
 several occasions. This was not just a onetime ploy: Kissinger made a policy
 of working with those who disagreed with him. Colleagues commented
 that he seemed to get along better with his enemies than with his friends.""",
        "qas" : [
            {
                "question" : "Do friends always mean what they say?",
                "id" : "1",

            }
        ],
    }
]

In [63]:
answers1, probabilities1 = model.predict(to_predict1)
print(answers1)

add example index and unique id: 100%|██████████| 1/1 [00:00<00:00, 2750.36it/s]


Running Prediction:   0%|          | 0/1 [00:00<?, ?it/s]

[{'id': '1', 'answer': ['friendliness (real or false) only obscures that fact. The key to power, then, is the ability to judge who is best able to further your interests in all situations. Keep friends for friendship, but work with the skilled and competent. Your enemies, on the other hand, are an untapped gold mine that you must learn to exploit. When Talley', 'make friends; friendliness (real or false) only obscures that fact. The key to power, then, is the ability to judge who is best able to further your interests in all situations. Keep friends for friendship, but work with the skilled and competent. Your enemies, on the other hand, are an untapped gold mine that you must learn to exploit. When Talley', 'War, Henry Kissinger was the target of an unsuccessful kidnapping attempt, a conspiracy involving, among']}]


In [72]:
to_predict2 = [
    {
        "context" : "Amazon markets AWS to subscribers as a way of obtaining large-scale computing capacity more quickly and cheaply than building an actual physical server farm. All services are billed based on usage, but each service measures usage in varying ways. As of 2023 Q1, AWS has 31% market share for cloud infrastructure while the next two competitors Microsoft Azure and Google Cloud have 25%, and 11% respectively, according to Synergy Research Group.",
        "qas" : [
            {
                "question" : "What is the market share of AWS?",
                "id" : "2",
            }
        ],
    }
]

In [73]:
answers2, probabilities2 = model.predict(to_predict2)
print(answers2)
print(probabilities2)

add example index and unique id: 100%|██████████| 1/1 [00:00<00:00, 4660.34it/s]


Running Prediction:   0%|          | 0/1 [00:00<?, ?it/s]

[{'id': '2', 'answer': ['usage in varying ways. As of 2023 Q1, AWS has 31% market share for cloud infrastructure', 'usage in varying ways. As of 2023 Q1, AWS has 31% market share for cloud infrastructure while the next two competitors Microsoft Azure']}]
[{'id': '2', 'probability': [0.38331115991179, 0.3716993648598752]}]
